In [ ]:
!pip install pyarrow pandas numpy scipy matplotlib networkx scikit-learn

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc

from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.sparse import coo_matrix, diags
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# Config
# ============================================================

SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path("datasets")
DATA_DIR.mkdir(exist_ok=True)

FILE_NAME = "connectome-weights-male-cns-v1.0-minconf-0.5.feather"
FILE_PATH = DATA_DIR / FILE_NAME
URL = f"https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/{FILE_NAME}"

TOP_EDGES = 30000
MAX_NEURONS = 200
SUBGRAPH_CACHE = DATA_DIR / f"male_cns_subgraph_n{MAX_NEURONS}_top{TOP_EDGES}.feather"

DT = 0.05
WHEEL_BASE = 2.7
MAX_STEER_RAD = np.deg2rad(30)
MAX_STEER_RATE = 0.06
MAX_ACCEL = 3.0
MAX_SPEED = 12.0
LANE_WIDTH = 3.5

INPUT_FEATURES = 5
TUNING_PER_FEATURE = 8

RECURRENT_GAIN = 0.50
INPUT_GAIN = 1.20

TRAIN_EPISODES = 20
TRAIN_STEPS = 900
WARMUP = 50

DAGGER_ROUNDS = 4
DAGGER_EPISODES = 6
DAGGER_STEPS = 800

TEST_STEPS = 1200
STEPS_PER_FRAME = 3

plt.rcParams["animation.embed_limit"] = 80


# ============================================================
# Dataset
# ============================================================

def download_dataset():
    if FILE_PATH.exists() and FILE_PATH.stat().st_size > 900_000_000:
        print(f"[OK] Dataset exists: {FILE_PATH}")
        return

    temp_path = FILE_PATH.with_suffix(".part")

    def progress(block_num, block_size, total_size):
        if total_size <= 0: return

        downloaded = block_num * block_size
        percent = min(downloaded / total_size * 100, 100)

        print(
            f"\rDownloading: {percent:6.2f}% "
            f"{downloaded / 1024**3:.2f}/{total_size / 1024**3:.2f} GB",
            end="",
        )

    print("Downloading MaleCNS...")
    urllib.request.urlretrieve(URL, temp_path, reporthook=progress)

    temp_path.replace(FILE_PATH)
    print("\n[OK] Download complete.")


# ============================================================
# Connectome
# ============================================================

def scan_top_edges(path, top_k):
    print("\nScanning strong connections...")

    source = pa.memory_map(str(path), "r")
    reader = ipc.open_file(source)

    columns = ["body_pre", "body_post", "weight"]
    best = None

    for i in range(reader.num_record_batches):
        df = reader.get_batch(i).select(columns).to_pandas()
        df = df.nlargest(min(top_k, len(df)), "weight")

        if best is None:
            best = df
        else:
            best = pd.concat([best, df], ignore_index=True)
            best = best.nlargest(top_k, "weight")

        print(f"\rBatch {i + 1}/{reader.num_record_batches}", end="")

    print()

    return best.reset_index(drop=True)


def select_neurons(edges):
    pre = edges.groupby("body_pre")["weight"].sum()
    post = edges.groupby("body_post")["weight"].sum()

    strength = pre.add(post, fill_value=0)
    return strength.nlargest(MAX_NEURONS).index.to_numpy()


def collect_internal_edges(path, selected):
    print("Collecting internal connections...")

    selected = set(map(int, selected))

    source = pa.memory_map(str(path), "r")
    reader = ipc.open_file(source)

    columns = ["body_pre", "body_post", "weight"]
    chunks = []

    for i in range(reader.num_record_batches):
        df = reader.get_batch(i).select(columns).to_pandas()

        mask = df["body_pre"].isin(selected) & df["body_post"].isin(selected)
        if mask.any(): chunks.append(df.loc[mask, columns])

        print(f"\rBatch {i + 1}/{reader.num_record_batches}", end="")

    print()

    sub = pd.concat(chunks, ignore_index=True)
    sub = sub.groupby(["body_pre", "body_post"], as_index=False)["weight"].sum()
    sub = sub[sub["body_pre"] != sub["body_post"]]

    return sub


def largest_component(sub):
    G = nx.DiGraph()

    for row in sub.itertuples():
        G.add_edge(int(row.body_pre), int(row.body_post))

    nodes = max(nx.weakly_connected_components(G), key=len)

    sub = sub[sub["body_pre"].isin(nodes) & sub["body_post"].isin(nodes)].copy()
    body_ids = sorted(nodes)

    return sub, body_ids


def load_subgraph():
    if SUBGRAPH_CACHE.exists():
        sub = pd.read_feather(SUBGRAPH_CACHE)
        body_ids = sorted(set(sub["body_pre"]).union(sub["body_post"]))

        print(f"[OK] Fly neurons: {len(body_ids)}")
        print(f"[OK] Fly edges: {len(sub)}")

        return sub, body_ids

    edges = scan_top_edges(FILE_PATH, TOP_EDGES)
    selected = select_neurons(edges)

    sub = collect_internal_edges(FILE_PATH, selected)
    sub, body_ids = largest_component(sub)

    sub.reset_index(drop=True).to_feather(SUBGRAPH_CACHE)

    print(f"[OK] Fly neurons: {len(body_ids)}")
    print(f"[OK] Fly edges: {len(sub)}")

    return sub, body_ids


def build_weight_matrix(sub, body_ids):
    id_to_idx = {body_id: i for i, body_id in enumerate(body_ids)}

    sub = sub.copy()

    sub["pre_idx"] = sub["body_pre"].map(id_to_idx)
    sub["post_idx"] = sub["body_post"].map(id_to_idx)

    row = sub["post_idx"].to_numpy()
    col = sub["pre_idx"].to_numpy()
    weight = np.log1p(sub["weight"].to_numpy(dtype=np.float32))

    N = len(body_ids)

    W = coo_matrix((weight, (row, col)), shape=(N, N)).tocsr()

    row_sum = np.asarray(W.sum(axis=1)).ravel()

    scale = np.zeros(N, dtype=np.float32)
    mask = row_sum > 0

    scale[mask] = RECURRENT_GAIN / row_sum[mask]

    return (diags(scale) @ W).tocsr(), sub


# ============================================================
# Fly Brain
# ============================================================

class FlyBrain:
    def __init__(self, W):
        self.W = W
        self.N = W.shape[0]

        self.threshold = 1.0
        self.voltage_decay = 0.88
        self.synaptic_decay = 0.40
        self.rate_decay = 0.95

        self.reset()

    def reset(self):
        self.V = np.zeros(self.N, dtype=np.float32)
        self.I = np.zeros(self.N, dtype=np.float32)
        self.spike = np.zeros(self.N, dtype=np.float32)
        self.rate = np.zeros(self.N, dtype=np.float32)

    def step(self, external):
        recurrent = self.W @ self.spike

        self.I = self.synaptic_decay * self.I + recurrent + external
        self.V = self.voltage_decay * self.V + self.I

        self.spike = (self.V >= self.threshold).astype(np.float32)
        self.V[self.spike > 0] = 0

        self.rate = self.rate_decay * self.rate + (1 - self.rate_decay) * self.spike

    def feature(self):
        return np.concatenate([
            self.rate,
            np.clip(self.V, 0, 2),
            np.clip(self.I, 0, 2),
        ])


# ============================================================
# Encoder
# ============================================================

class FlyEncoder:
    def __init__(self, W):
        count = INPUT_FEATURES * TUNING_PER_FEATURE

        if W.shape[0] < count: raise RuntimeError("Not enough neurons.")

        strength = np.asarray(abs(W).sum(axis=0)).ravel()

        self.N = W.shape[0]
        self.nodes = np.argsort(-strength)[:count]

        self.centers = np.linspace(-1, 1, TUNING_PER_FEATURE)
        self.sigma = 0.30

    def encode(self, observation):
        cte, heading_error, curvature, speed, target_angle = observation

        values = np.array([
            np.clip(cte / 4.0, -1, 1),
            np.clip(heading_error / 1.0, -1, 1),
            np.clip(curvature / 0.08, -1, 1),
            np.clip((speed - 6) / 6, -1, 1),
            np.clip(target_angle / 1.5, -1, 1),
        ])

        external = np.zeros(self.N, dtype=np.float32)

        for i, value in enumerate(values):
            activation = np.exp(-0.5 * ((value - self.centers) / self.sigma) ** 2)

            start = i * TUNING_PER_FEATURE
            nodes = self.nodes[start:start + TUNING_PER_FEATURE]

            external[nodes] = INPUT_GAIN * activation

        return external


# ============================================================
# Road
# ============================================================

class Road:
    def __init__(self, a1=2.5, a2=1.2, f1=0.025, f2=0.055, p1=0, p2=0):
        self.a1 = a1; self.a2 = a2
        self.f1 = f1; self.f2 = f2
        self.p1 = p1; self.p2 = p2

    def y(self, x):
        return self.a1 * np.sin(self.f1 * x + self.p1) + self.a2 * np.sin(self.f2 * x + self.p2)

    def dy(self, x):
        return self.a1 * self.f1 * np.cos(self.f1 * x + self.p1) + self.a2 * self.f2 * np.cos(self.f2 * x + self.p2)

    def ddy(self, x):
        return -self.a1 * self.f1**2 * np.sin(self.f1 * x + self.p1) - self.a2 * self.f2**2 * np.sin(self.f2 * x + self.p2)

    def heading(self, x):
        return np.arctan2(self.dy(x), 1)

    def curvature(self, x):
        dy = self.dy(x)
        return self.ddy(x) / (1 + dy**2) ** 1.5


def random_road(generator, harder=False):
    if harder:
        return Road(
            a1=generator.uniform(2.0, 4.5),
            a2=generator.uniform(0.8, 2.2),
            f1=generator.uniform(0.018, 0.040),
            f2=generator.uniform(0.045, 0.085),
            p1=generator.uniform(0, 2 * np.pi),
            p2=generator.uniform(0, 2 * np.pi),
        )

    return Road(
        a1=generator.uniform(1.5, 4.0),
        a2=generator.uniform(0.5, 2.0),
        f1=generator.uniform(0.015, 0.035),
        f2=generator.uniform(0.040, 0.075),
        p1=generator.uniform(0, 2 * np.pi),
        p2=generator.uniform(0, 2 * np.pi),
    )


# ============================================================
# Vehicle
# ============================================================

def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


class Vehicle:
    def __init__(self, road, y_offset=0, yaw_offset=0):
        self.x = 0.0
        self.y = road.y(0) + y_offset
        self.yaw = road.heading(0) + yaw_offset
        self.v = 5.0

    def step(self, steering, throttle):
        steering = np.clip(steering, -1, 1)
        throttle = np.clip(throttle, -1, 1)

        delta = steering * MAX_STEER_RAD

        self.x += self.v * np.cos(self.yaw) * DT
        self.y += self.v * np.sin(self.yaw) * DT
        self.yaw = wrap_angle(self.yaw + self.v / WHEEL_BASE * np.tan(delta) * DT)

        acceleration = throttle * MAX_ACCEL - 0.025 * self.v**2

        self.v += acceleration * DT
        self.v = np.clip(self.v, 1.5, MAX_SPEED)


# ============================================================
# Observation
# ============================================================

def observe(vehicle, road):
    road_y = road.y(vehicle.x)
    road_heading = road.heading(vehicle.x)

    cte = (vehicle.y - road_y) * np.cos(road_heading)
    heading_error = wrap_angle(road_heading - vehicle.yaw)

    lookahead = np.clip(6 + 0.5 * vehicle.v, 6, 12)

    target_x = vehicle.x + lookahead
    target_y = road.y(target_x)

    target_angle = wrap_angle(
        np.arctan2(target_y - vehicle.y, target_x - vehicle.x) - vehicle.yaw
    )

    curvature = road.curvature(vehicle.x + lookahead)

    return np.array([
        cte,
        heading_error,
        curvature,
        vehicle.v,
        target_angle,
    ], dtype=np.float32)


# ============================================================
# Robust Teacher - Pure Pursuit
# ============================================================

def teacher_steering(vehicle, road):
    lookahead = np.clip(6 + 0.5 * vehicle.v, 6, 12)

    target_x = vehicle.x + lookahead
    target_y = road.y(target_x)

    alpha = wrap_angle(
        np.arctan2(target_y - vehicle.y, target_x - vehicle.x) - vehicle.yaw
    )

    delta = np.arctan2(
        2 * WHEEL_BASE * np.sin(alpha),
        lookahead,
    )

    return np.clip(delta / MAX_STEER_RAD, -1, 1)


def speed_controller(observation):
    cte, heading_error, curvature, speed, _ = observation

    target_speed = 10.5 / (1 + 25 * abs(curvature))

    recovery = 1 - 0.12 * abs(cte) - 0.30 * abs(heading_error)
    recovery = np.clip(recovery, 0.35, 1.0)

    target_speed *= recovery
    target_speed = np.clip(target_speed, 3.5, 10.5)

    return np.clip(0.45 * (target_speed - speed), -1, 1)


# ============================================================
# Initial Teacher Dataset
# ============================================================

def collect_teacher_data(brain, encoder, episodes, steps, generator):
    X, Y = [], []

    for episode in range(episodes):
        road = random_road(generator)

        vehicle = Vehicle(
            road,
            y_offset=generator.uniform(-2.5, 2.5),
            yaw_offset=generator.uniform(-0.40, 0.40),
        )

        brain.reset()

        for t in range(steps):
            observation = observe(vehicle, road)

            brain.step(encoder.encode(observation))

            steering = teacher_steering(vehicle, road)
            throttle = speed_controller(observation)

            if t >= WARMUP:
                X.append(brain.feature().copy())
                Y.append(steering)

            vehicle.step(steering, throttle)

        print(f"Teacher episode {episode + 1}/{episodes}")

    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)


# ============================================================
# Model
# ============================================================

def train_model(X, Y):
    model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-2, 3, 12)),
    )

    model.fit(X, Y)

    pred = model.predict(X)
    rmse = np.sqrt(np.mean((pred - Y) ** 2))

    print(f"Samples: {len(X)}")
    print(f"Train RMSE: {rmse:.4f}")
    print(f"Alpha: {model.named_steps['ridgecv'].alpha_}")

    return model


def predict_steering(model, feature):
    raw = model.predict(feature[None])[0]

    # clip보다 부드럽게 포화
    return float(np.tanh(raw))


# ============================================================
# DAgger
# ============================================================

def collect_dagger_data(model, brain, encoder, episodes, steps, generator, beta):
    X, Y = [], []

    for episode in range(episodes):
        road = random_road(generator, harder=True)

        vehicle = Vehicle(
            road,
            y_offset=generator.uniform(-3.0, 3.0),
            yaw_offset=generator.uniform(-0.55, 0.55),
        )

        brain.reset()
        steering = 0.0

        for t in range(steps):
            observation = observe(vehicle, road)

            brain.step(encoder.encode(observation))
            feature = brain.feature()

            teacher = teacher_steering(vehicle, road)
            predicted = predict_steering(model, feature)

            # teacher와 현재 모델을 섞어 모델이 실제로 갈 법한 상태를 생성
            desired = beta * teacher + (1 - beta) * predicted

            diff = np.clip(desired - steering, -MAX_STEER_RATE, MAX_STEER_RATE)
            steering += diff

            throttle = speed_controller(observation)

            if t >= WARMUP:
                X.append(feature.copy())
                Y.append(teacher)

            vehicle.step(steering, throttle)

        print(f"DAgger β={beta:.2f} | episode {episode + 1}/{episodes}")

    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)


def dagger_train(brain, encoder):
    generator = np.random.default_rng(SEED)

    print("\n==============================")
    print("INITIAL TEACHER DATA")
    print("==============================")

    X, Y = collect_teacher_data(
        brain,
        encoder,
        TRAIN_EPISODES,
        TRAIN_STEPS,
        generator,
    )

    model = train_model(X, Y)

    betas = [0.7, 0.4, 0.2, 0.0]

    for round_idx, beta in enumerate(betas):
        print("\n==============================")
        print(f"DAGGER {round_idx + 1}/{len(betas)}")
        print("==============================")

        X_new, Y_new = collect_dagger_data(
            model,
            brain,
            encoder,
            DAGGER_EPISODES,
            DAGGER_STEPS,
            generator,
            beta,
        )

        X = np.concatenate([X, X_new])
        Y = np.concatenate([Y, Y_new])

        model = train_model(X, Y)

    return model


# ============================================================
# Autonomous Test
# ============================================================

def simulate(model, brain, encoder, road, steps, y_offset=1.5, yaw_offset=-0.2):
    vehicle = Vehicle(road, y_offset, yaw_offset)

    brain.reset()
    steering = 0.0

    history = {
        "x": [],
        "y": [],
        "yaw": [],
        "speed": [],
        "cte": [],
        "heading": [],
        "steering": [],
        "throttle": [],
        "rate": [],
        "spike": [],
    }

    for _ in range(steps):
        observation = observe(vehicle, road)

        brain.step(encoder.encode(observation))

        desired = predict_steering(model, brain.feature())

        # 갑자기 ±1로 튀는 현상 방지
        diff = np.clip(desired - steering, -MAX_STEER_RATE, MAX_STEER_RATE)
        steering += diff

        throttle = speed_controller(observation)

        vehicle.step(steering, throttle)

        history["x"].append(vehicle.x)
        history["y"].append(vehicle.y)
        history["yaw"].append(vehicle.yaw)
        history["speed"].append(vehicle.v)
        history["cte"].append(observation[0])
        history["heading"].append(observation[1])
        history["steering"].append(steering)
        history["throttle"].append(throttle)
        history["rate"].append(brain.rate.copy())
        history["spike"].append(brain.spike.copy())

    return {key: np.asarray(value) for key, value in history.items()}


def test_model(model, brain, encoder, episodes=8):
    generator = np.random.default_rng(777)

    means = []
    departures = []

    print("\n==============================")
    print("CLOSED LOOP TEST")
    print("==============================")

    for i in range(episodes):
        road = random_road(generator, harder=True)

        history = simulate(
            model,
            brain,
            encoder,
            road,
            1000,
            y_offset=generator.uniform(-2, 2),
            yaw_offset=generator.uniform(-0.3, 0.3),
        )

        cte = np.abs(history["cte"])

        mean_cte = cte.mean()
        departure = np.mean(cte > LANE_WIDTH / 2)

        means.append(mean_cte)
        departures.append(departure)

        print(
            f"Test {i + 1}: "
            f"mean|CTE|={mean_cte:.3f} m, "
            f"max={cte.max():.3f} m, "
            f"departure={departure * 100:.1f}%"
        )

    print(f"\nAverage mean |CTE|: {np.mean(means):.3f} m")
    print(f"Average departure: {np.mean(departures) * 100:.2f}%")


# ============================================================
# Graph
# ============================================================

def build_graph(sub, body_ids):
    id_to_idx = {body_id: i for i, body_id in enumerate(body_ids)}

    G = nx.DiGraph()
    G.add_nodes_from(range(len(body_ids)))

    for row in sub.itertuples():
        G.add_edge(id_to_idx[row.body_pre], id_to_idx[row.body_post])

    pos = nx.spring_layout(G, seed=SEED, k=0.25, iterations=80)

    return G, pos


# ============================================================
# Animation
# ============================================================

def animate(history, road, brain, encoder, G, pos):
    frame_ids = np.arange(0, len(history["x"]), STEPS_PER_FRAME)

    x_road = np.linspace(-20, max(100, history["x"].max() + 50), 2500)
    center = road.y(x_road)
    theta = road.heading(x_road)

    left_x = x_road - LANE_WIDTH / 2 * np.sin(theta)
    left_y = center + LANE_WIDTH / 2 * np.cos(theta)

    right_x = x_road + LANE_WIDTH / 2 * np.sin(theta)
    right_y = center - LANE_WIDTH / 2 * np.cos(theta)

    fig, (ax_road, ax_brain) = plt.subplots(1, 2, figsize=(14, 6))

    ax_road.plot(left_x, left_y, linewidth=2)
    ax_road.plot(right_x, right_y, linewidth=2)
    ax_road.plot(x_road, center, "--", linewidth=1)

    car, = ax_road.plot([], [], "o", markersize=10)
    direction, = ax_road.plot([], [], linewidth=2)
    trail, = ax_road.plot([], [], linewidth=1)

    ax_road.set_xlabel("x [m]"); ax_road.set_ylabel("y [m]")
    ax_road.grid(True); ax_road.set_aspect("equal")

    road_text = ax_road.text(0.02, 0.98, "", transform=ax_road.transAxes, va="top")

    edges = list(G.edges())

    if len(edges) > 1500:
        idx = np.random.default_rng(SEED).choice(len(edges), 1500, replace=False)
        edges = [edges[i] for i in idx]

    nx.draw_networkx_edges(G, pos, edgelist=edges, ax=ax_brain, arrows=False, alpha=0.06, width=0.5)

    coords = np.array([pos[i] for i in range(brain.N)])

    nodes = ax_brain.scatter(
        coords[:, 0],
        coords[:, 1],
        s=25,
        c=np.zeros(brain.N),
        cmap="plasma",
        vmin=0,
        vmax=0.25,
    )

    inputs = coords[encoder.nodes]

    ax_brain.scatter(
        inputs[:, 0],
        inputs[:, 1],
        s=55,
        marker="s",
        facecolors="none",
        edgecolors="black",
        label="Sensor neurons",
    )

    ax_brain.set_title("MaleCNS Reservoir Activity")
    ax_brain.axis("off"); ax_brain.legend()

    brain_text = ax_brain.text(0.02, 0.98, "", transform=ax_brain.transAxes, va="top")

    def update(frame):
        i = frame_ids[frame]

        x = history["x"][i]
        y = history["y"][i]
        yaw = history["yaw"][i]

        car.set_data([x], [y])
        direction.set_data([x, x + 3 * np.cos(yaw)], [y, y + 3 * np.sin(yaw)])

        start = max(0, i - 250)
        trail.set_data(history["x"][start:i + 1], history["y"][start:i + 1])

        ax_road.set_xlim(x - 10, x + 50)

        middle = road.y(x + 20)
        ax_road.set_ylim(middle - 12, middle + 12)

        mean_cte = np.mean(np.abs(history["cte"][max(0, i - 100):i + 1]))

        road_text.set_text(
            f"t = {i * DT:.1f} s\n"
            f"speed = {history['speed'][i]:.2f} m/s\n"
            f"CTE = {history['cte'][i]:+.2f} m\n"
            f"heading = {np.rad2deg(history['heading'][i]):+.1f}°\n"
            f"steering = {history['steering'][i]:+.2f}\n"
            f"mean |CTE| = {mean_cte:.2f} m"
        )

        rate = history["rate"][i]
        spike = history["spike"][i]

        nodes.set_array(rate)
        nodes.set_sizes(25 + 100 * spike)

        brain_text.set_text(
            f"spikes = {int(spike.sum())}\n"
            f"mean activity = {rate.mean():.4f}\n"
            f"active neurons = {np.sum(rate > 0.02)}/{brain.N}"
        )

        return car, direction, trail, nodes, road_text, brain_text

    anim = FuncAnimation(
        fig,
        update,
        frames=len(frame_ids),
        interval=int(DT * STEPS_PER_FRAME * 1000),
        blit=False,
    )

    plt.tight_layout()
    plt.close(fig)

    display(HTML(anim.to_jshtml(default_mode="loop")))

    return anim


# ============================================================
# Main
# ============================================================

download_dataset()

sub_edges, body_ids = load_subgraph()
W, sub_edges = build_weight_matrix(sub_edges, body_ids)

print(f"W shape: {W.shape}")
print(f"W connections: {W.nnz}")

brain = FlyBrain(W)
encoder = FlyEncoder(W)


# ============================================================
# Train
# ============================================================

model = dagger_train(brain, encoder)


# ============================================================
# Test
# ============================================================

test_model(model, brain, encoder)


# ============================================================
# Final unseen road
# ============================================================

test_road = Road(
    a1=3.4,
    a2=1.5,
    f1=0.026,
    f2=0.061,
    p1=1.2,
    p2=2.5,
)

history = simulate(
    model,
    brain,
    encoder,
    test_road,
    TEST_STEPS,
    y_offset=1.5,
    yaw_offset=-0.18,
)

cte = np.abs(history["cte"])

print("\n==============================")
print("FINAL ROAD")
print("==============================")

print(f"Mean |CTE|: {cte.mean():.3f} m")
print(f"Max |CTE|: {cte.max():.3f} m")
print(f"Lane departure: {np.mean(cte > LANE_WIDTH / 2) * 100:.2f}%")


# ============================================================
# Visualization
# ============================================================

G, pos = build_graph(sub_edges, body_ids)
anim = animate(history, test_road, brain, encoder, G, pos)

In [ ]:
!pip install imageio_ffmpeg

In [ ]:
import urllib.request
from pathlib import Path

import imageio_ffmpeg
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc

from IPython.display import Video, display
from matplotlib.animation import FFMpegWriter, FuncAnimation
from scipy.sparse import coo_matrix, load_npz, save_npz
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# Config
# ============================================================

SEED = 42

DATA_DIR = Path("datasets")
OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

FILE_NAME = "connectome-weights-male-cns-v1.0-minconf-0.5.feather"
FILE_PATH = DATA_DIR / FILE_NAME

URL = (
    "https://storage.googleapis.com/flyem-male-cns/"
    f"v1.0/connectome-data/flat-connectome/{FILE_NAME}"
)

# 1 = 모든 edge
# 3 = 약한 edge 제거, 뉴런은 전체 사용
MIN_SYNAPSES = 3

BODY_CACHE = DATA_DIR / "malecns_all_body_ids.npy"
GRAPH_CACHE = DATA_DIR / f"malecns_full_w{MIN_SYNAPSES}.npz"


# ============================================================
# Vehicle
# ============================================================

DT = 0.05

WHEEL_BASE = 2.7
MAX_STEER_RAD = np.deg2rad(30)
MAX_STEER_RATE = 0.07

MAX_ACCEL = 3.0
MAX_SPEED = 11.0

LANE_WIDTH = 3.5


# ============================================================
# Fly Brain
# ============================================================

INPUT_FEATURES = 5
TUNING_PER_FEATURE = 8

RECURRENT_GAIN = 0.55
INPUT_GAIN = 1.25

THRESHOLD = 1.0

VOLTAGE_DECAY = 0.88
SYNAPTIC_DECAY = 0.40
RATE_DECAY = 0.95

MAX_SPIKE_FRACTION = 0.005

READOUT_DIM = 256


# ============================================================
# Training
# ============================================================

TRAIN_EPISODES = 5
TRAIN_STEPS = 450

DAGGER_ROUNDS = 2
DAGGER_EPISODES = 3
DAGGER_STEPS = 350

WARMUP = 30


# ============================================================
# Test / Video
# ============================================================

TEST_STEPS = 900

DISPLAY_NEURONS = 350
DISPLAY_EDGES = 1200
TOP_STEERING_NEURONS = 35

STEPS_PER_FRAME = 3

VIDEO_PATH = OUTPUT_DIR / "malecns_aggressive_vertical.mp4"

plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()


# ============================================================
# Dataset
# ============================================================

def download_dataset():
    if FILE_PATH.exists():
        print(f"[OK] Dataset exists: {FILE_PATH}")
        return

    temp_path = FILE_PATH.with_suffix(".part")

    def progress(block_num, block_size, total_size):
        if total_size <= 0: return

        downloaded = block_num * block_size
        percent = min(downloaded / total_size * 100, 100)

        print(
            f"\rDownloading: {percent:6.2f}% "
            f"{downloaded / 1024**3:.2f}/{total_size / 1024**3:.2f} GB",
            end="",
        )

    print("Downloading MaleCNS...")

    urllib.request.urlretrieve(URL, temp_path, reporthook=progress)
    temp_path.replace(FILE_PATH)

    print("\n[OK] Download complete.")


# ============================================================
# Full Neuron IDs
# ============================================================

def scan_all_body_ids():
    if BODY_CACHE.exists():
        body_ids = np.load(BODY_CACHE)

        print(f"[OK] Neuron IDs: {len(body_ids):,}")

        return body_ids

    print("\nScanning all MaleCNS neuron IDs...")

    source = pa.memory_map(str(FILE_PATH), "r")
    reader = ipc.open_file(source)

    body_set = set()

    for i in range(reader.num_record_batches):
        batch = reader.get_batch(i).select(["body_pre", "body_post"])

        pre = batch.column("body_pre").to_numpy()
        post = batch.column("body_post").to_numpy()

        body_set.update(np.unique(pre).tolist())
        body_set.update(np.unique(post).tolist())

        print(
            f"\rNeuron scan {i + 1}/{reader.num_record_batches}",
            end="",
        )

    print()

    body_ids = np.array(
        sorted(body_set),
        dtype=np.int64,
    )

    np.save(BODY_CACHE, body_ids)

    print(f"[OK] Full neurons: {len(body_ids):,}")

    return body_ids


# ============================================================
# Full Connectome
# ============================================================

def build_full_connectome(body_ids):
    if GRAPH_CACHE.exists():
        print(f"\n[OK] Loading graph cache: {GRAPH_CACHE}")

        W = load_npz(GRAPH_CACHE).tocsc()

        print(f"Neurons: {W.shape[0]:,}")
        print(f"Connections: {W.nnz:,}")

        return W

    print("\nBuilding full MaleCNS connectivity...")
    print(f"MIN_SYNAPSES = {MIN_SYNAPSES}")

    source = pa.memory_map(str(FILE_PATH), "r")
    reader = ipc.open_file(source)

    rows = []
    cols = []
    values = []

    total_edges = 0

    for i in range(reader.num_record_batches):
        batch = reader.get_batch(i).select(
            ["body_pre", "body_post", "weight"]
        )

        df = batch.to_pandas()

        if MIN_SYNAPSES > 1:
            df = df[df["weight"] >= MIN_SYNAPSES]

        if len(df) == 0: continue

        pre = df["body_pre"].to_numpy(dtype=np.int64, copy=False)
        post = df["body_post"].to_numpy(dtype=np.int64, copy=False)
        weight = df["weight"].to_numpy(dtype=np.float32, copy=False)

        pre_idx = np.searchsorted(body_ids, pre).astype(np.int32)
        post_idx = np.searchsorted(body_ids, post).astype(np.int32)

        cols.append(pre_idx)
        rows.append(post_idx)
        values.append(np.log1p(weight).astype(np.float32))

        total_edges += len(df)

        print(
            f"\rEdge scan {i + 1}/{reader.num_record_batches} "
            f"| kept={total_edges:,}",
            end="",
        )

    print("\nCreating sparse graph...")

    row = np.concatenate(rows)
    col = np.concatenate(cols)
    data = np.concatenate(values)

    del rows, cols, values

    N = len(body_ids)

    W = coo_matrix(
        (data, (row, col)),
        shape=(N, N),
        dtype=np.float32,
    ).tocsr()

    del row, col, data

    W.sum_duplicates()

    print("Normalizing graph...")

    row_sum = np.asarray(
        W.sum(axis=1)
    ).ravel().astype(np.float32)

    scale = np.zeros(
        N,
        dtype=np.float32,
    )

    mask = row_sum > 0
    scale[mask] = RECURRENT_GAIN / row_sum[mask]

    for i in range(N):
        start = W.indptr[i]
        end = W.indptr[i + 1]

        if start != end:
            W.data[start:end] *= scale[i]

    W = W.tocsc()

    print(f"Full neurons: {W.shape[0]:,}")
    print(f"Connections: {W.nnz:,}")

    save_npz(GRAPH_CACHE, W, compressed=False)

    print(f"[OK] Cached: {GRAPH_CACHE}")

    return W


# ============================================================
# Fly Brain
# ============================================================

class FlyBrain:
    def __init__(self, W):
        self.W = W
        self.N = W.shape[0]

        self.max_spikes = max(
            1,
            int(self.N * MAX_SPIKE_FRACTION),
        )

        self.reset()

    def reset(self):
        self.V = np.zeros(self.N, dtype=np.float32)
        self.I = np.zeros(self.N, dtype=np.float32)
        self.spike = np.zeros(self.N, dtype=np.float32)
        self.rate = np.zeros(self.N, dtype=np.float32)

    def recurrent_current(self):
        active = np.flatnonzero(self.spike)

        recurrent = np.zeros(
            self.N,
            dtype=np.float32,
        )

        for neuron in active:
            start = self.W.indptr[neuron]
            end = self.W.indptr[neuron + 1]

            targets = self.W.indices[start:end]
            weights = self.W.data[start:end]

            np.add.at(
                recurrent,
                targets,
                weights,
            )

        return recurrent

    def step(self, external):
        recurrent = self.recurrent_current()

        global_inhibition = 0.20 * self.spike.mean()

        self.I = (
            SYNAPTIC_DECAY * self.I
            + recurrent
            + external
            - global_inhibition
        )

        self.I = np.clip(
            self.I,
            -2,
            2,
        )

        self.V = (
            VOLTAGE_DECAY * self.V
            + self.I
        )

        candidates = np.flatnonzero(
            self.V >= THRESHOLD
        )

        self.spike.fill(0)

        if len(candidates) > self.max_spikes:
            value = self.V[candidates]

            idx = np.argpartition(
                value,
                -self.max_spikes,
            )[-self.max_spikes:]

            candidates = candidates[idx]

        self.spike[candidates] = 1
        self.V[candidates] = 0

        self.rate = (
            RATE_DECAY * self.rate
            + (1 - RATE_DECAY) * self.spike
        )


# ============================================================
# Encoder
# ============================================================

class FlyEncoder:
    def __init__(self, W):
        self.N = W.shape[0]

        count = INPUT_FEATURES * TUNING_PER_FEATURE

        out_degree = np.diff(W.indptr)

        self.nodes = np.argsort(
            -out_degree
        )[:count]

        self.centers = np.linspace(
            -1,
            1,
            TUNING_PER_FEATURE,
        )

        self.sigma = 0.28

    def normalize(self, observation):
        cte, heading_error, curvature, speed, target_angle = observation

        return np.array([
            np.clip(cte / 4.0, -1, 1),
            np.clip(heading_error / 1.2, -1, 1),
            np.clip(curvature / 0.14, -1, 1),
            np.clip((speed - 6.0) / 6.0, -1, 1),
            np.clip(target_angle / 1.5, -1, 1),
        ], dtype=np.float32)

    def encode(self, observation):
        values = self.normalize(observation)

        external = np.zeros(
            self.N,
            dtype=np.float32,
        )

        for i, value in enumerate(values):
            activation = np.exp(
                -0.5
                * (
                    (value - self.centers)
                    / self.sigma
                )
                ** 2
            )

            start = i * TUNING_PER_FEATURE
            nodes = self.nodes[start:start + TUNING_PER_FEATURE]

            external[nodes] = INPUT_GAIN * activation

        return external


# ============================================================
# Whole Brain Projection
# ============================================================

class BrainProjector:
    def __init__(self, N, dim=READOUT_DIM):
        generator = np.random.default_rng(2026)

        self.dim = dim

        self.bucket = generator.integers(
            0,
            dim,
            size=N,
            dtype=np.int32,
        )

        self.sign = generator.choice(
            [-1.0, 1.0],
            size=N,
        ).astype(np.float32)

        count = np.bincount(
            self.bucket,
            minlength=dim,
        ).astype(np.float32)

        self.norm = np.sqrt(
            np.maximum(count, 1)
        )

    def project(self, x):
        out = np.bincount(
            self.bucket,
            weights=x * self.sign,
            minlength=self.dim,
        ).astype(np.float32)

        return out / self.norm


def make_feature(brain, projector):
    rate = projector.project(brain.rate)
    spike = projector.project(brain.spike)

    return np.concatenate([
        rate,
        spike,
    ])


# ============================================================
# Track
# ============================================================

class Track:
    def __init__(self, curvature, ds=0.25):
        self.ds = ds

        self.curvature = np.asarray(
            curvature,
            dtype=np.float32,
        )

        self.N = len(curvature)

        self.s = np.arange(
            self.N
        ) * ds

        self.x = np.zeros(self.N)
        self.y = np.zeros(self.N)
        self.yaw = np.zeros(self.N)

        for i in range(self.N - 1):
            kappa = self.curvature[i]

            mid_yaw = (
                self.yaw[i]
                + 0.5 * kappa * ds
            )

            self.x[i + 1] = (
                self.x[i]
                + ds * np.cos(mid_yaw)
            )

            self.y[i + 1] = (
                self.y[i]
                + ds * np.sin(mid_yaw)
            )

            self.yaw[i + 1] = (
                self.yaw[i]
                + kappa * ds
            )

        self.yaw = np.unwrap(
            self.yaw
        )

    def nearest_index(self, x, y, hint=0):
        lo = max(
            0,
            hint - 40,
        )

        hi = min(
            self.N,
            hint + 180,
        )

        dx = self.x[lo:hi] - x
        dy = self.y[lo:hi] - y

        idx = np.argmin(
            dx**2 + dy**2
        )

        return lo + idx

    def lookahead_index(self, index, distance):
        target_s = (
            self.s[index]
            + distance
        )

        idx = np.searchsorted(
            self.s,
            target_s,
        )

        return min(
            idx,
            self.N - 1,
        )


# ============================================================
# Random Training Track
# ============================================================

def random_track(generator, aggressive=False):
    length = 260.0
    ds = 0.25

    s = np.arange(
        0,
        length,
        ds,
    )

    if aggressive:
        base_amp = generator.uniform(0.010, 0.025)
        pulse_amp = 0.105
        turns = 6
    else:
        base_amp = generator.uniform(0.005, 0.015)
        pulse_amp = 0.070
        turns = 5

    curvature = (
        base_amp
        * np.sin(
            2 * np.pi * s
            / generator.uniform(35, 60)
            + generator.uniform(0, 2 * np.pi)
        )
    )

    curvature += (
        0.008
        * np.sin(
            2 * np.pi * s
            / generator.uniform(15, 28)
            + generator.uniform(0, 2 * np.pi)
        )
    )

    centers = np.linspace(
        30,
        length - 30,
        turns,
    )

    centers += generator.uniform(
        -8,
        8,
        size=turns,
    )

    for i, center in enumerate(centers):
        direction = (
            1
            if i % 2 == 0
            else -1
        )

        if generator.random() < 0.35:
            direction *= -1

        amplitude = (
            direction
            * generator.uniform(
                pulse_amp * 0.65,
                pulse_amp,
            )
        )

        width = generator.uniform(
            5.0,
            9.0,
        )

        curvature += (
            amplitude
            * np.exp(
                -0.5
                * (
                    (s - center)
                    / width
                )
                ** 2
            )
        )

    limit = (
        0.13
        if aggressive
        else 0.085
    )

    curvature = np.clip(
        curvature,
        -limit,
        limit,
    )

    ramp = np.clip(
        s / 15.0,
        0,
        1,
    )

    curvature *= ramp

    return Track(
        curvature,
        ds,
    )


# ============================================================
# Aggressive Final Track
# ============================================================

def aggressive_test_track():
    length = 280.0
    ds = 0.25

    s = np.arange(
        0,
        length,
        ds,
    )

    curvature = (
        0.015
        * np.sin(
            2 * np.pi * s / 38
        )
        + 0.008
        * np.sin(
            2 * np.pi * s / 18
        )
    )

    corners = [
        (35, +0.090, 8.0),
        (68, -0.125, 6.5),
        (100, +0.110, 7.0),
        (132, -0.130, 6.5),
        (165, +0.120, 7.5),
        (200, -0.115, 7.0),
        (235, +0.125, 6.5),
    ]

    for center, amplitude, width in corners:
        curvature += (
            amplitude
            * np.exp(
                -0.5
                * (
                    (s - center)
                    / width
                )
                ** 2
            )
        )

    curvature = np.clip(
        curvature,
        -0.14,
        0.14,
    )

    curvature *= np.clip(
        s / 12,
        0,
        1,
    )

    return Track(
        curvature,
        ds,
    )


# ============================================================
# Vehicle
# ============================================================

def wrap_angle(angle):
    return (
        angle + np.pi
    ) % (2 * np.pi) - np.pi


class Vehicle:
    def __init__(self, track, lateral_offset=0.0, yaw_offset=0.0):
        yaw = track.yaw[0]

        nx = -np.sin(yaw)
        ny = np.cos(yaw)

        self.x = (
            track.x[0]
            + lateral_offset * nx
        )

        self.y = (
            track.y[0]
            + lateral_offset * ny
        )

        self.yaw = (
            yaw
            + yaw_offset
        )

        self.v = 5.0

        self.track_idx = 0

    def step(self, steering, throttle):
        steering = np.clip(
            steering,
            -1,
            1,
        )

        throttle = np.clip(
            throttle,
            -1,
            1,
        )

        delta = (
            steering
            * MAX_STEER_RAD
        )

        self.x += (
            self.v
            * np.cos(self.yaw)
            * DT
        )

        self.y += (
            self.v
            * np.sin(self.yaw)
            * DT
        )

        self.yaw += (
            self.v
            / WHEEL_BASE
            * np.tan(delta)
            * DT
        )

        self.yaw = wrap_angle(
            self.yaw
        )

        acceleration = (
            throttle * MAX_ACCEL
            - 0.025 * self.v**2
        )

        self.v += (
            acceleration * DT
        )

        self.v = np.clip(
            self.v,
            2.0,
            MAX_SPEED,
        )


# ============================================================
# Observation
# ============================================================

def observe(vehicle, track):
    idx = track.nearest_index(
        vehicle.x,
        vehicle.y,
        vehicle.track_idx,
    )

    vehicle.track_idx = max(
        vehicle.track_idx,
        idx,
    )

    idx = vehicle.track_idx

    ref_x = track.x[idx]
    ref_y = track.y[idx]
    ref_yaw = track.yaw[idx]

    dx = vehicle.x - ref_x
    dy = vehicle.y - ref_y

    normal_x = -np.sin(ref_yaw)
    normal_y = np.cos(ref_yaw)

    cte = (
        dx * normal_x
        + dy * normal_y
    )

    heading_error = wrap_angle(
        ref_yaw
        - vehicle.yaw
    )

    lookahead = np.clip(
        4.5
        + 0.45 * vehicle.v,
        5.0,
        9.0,
    )

    target_idx = track.lookahead_index(
        idx,
        lookahead,
    )

    target_x = track.x[target_idx]
    target_y = track.y[target_idx]

    target_angle = wrap_angle(
        np.arctan2(
            target_y - vehicle.y,
            target_x - vehicle.x,
        )
        - vehicle.yaw
    )

    curvature = track.curvature[
        target_idx
    ]

    observation = np.array([
        cte,
        heading_error,
        curvature,
        vehicle.v,
        target_angle,
    ], dtype=np.float32)

    done = (
        idx
        >= track.N - 8
    )

    return observation, done


# ============================================================
# Teacher
# ============================================================

def teacher_steering(observation):
    _, _, _, speed, target_angle = observation

    lookahead = np.clip(
        4.5
        + 0.45 * speed,
        5,
        9,
    )

    delta = np.arctan2(
        2
        * WHEEL_BASE
        * np.sin(target_angle),
        lookahead,
    )

    return np.clip(
        delta / MAX_STEER_RAD,
        -1,
        1,
    )


def speed_controller(observation):
    cte, heading_error, curvature, speed, target_angle = observation

    target_speed = (
        10.0
        / (
            1
            + 35 * abs(curvature)
            + abs(target_angle)
        )
    )

    recovery = (
        1
        - 0.14 * abs(cte)
        - 0.30 * abs(heading_error)
    )

    recovery = np.clip(
        recovery,
        0.30,
        1.0,
    )

    target_speed *= recovery

    target_speed = np.clip(
        target_speed,
        3.0,
        10.0,
    )

    return np.clip(
        0.50
        * (
            target_speed
            - speed
        ),
        -1,
        1,
    )


# ============================================================
# Teacher Training Data
# ============================================================

def collect_teacher_data(brain, projector, encoder, episodes, steps, generator):
    X, Y = [], []

    for episode in range(episodes):
        track = random_track(
            generator,
            aggressive=False,
        )

        vehicle = Vehicle(
            track,
            lateral_offset=generator.uniform(-2.2, 2.2),
            yaw_offset=generator.uniform(-0.35, 0.35),
        )

        brain.reset()

        for t in range(steps):
            observation, done = observe(
                vehicle,
                track,
            )

            brain.step(
                encoder.encode(
                    observation
                )
            )

            steering = teacher_steering(
                observation
            )

            throttle = speed_controller(
                observation
            )

            if t >= WARMUP:
                X.append(
                    make_feature(
                        brain,
                        projector,
                    )
                )

                Y.append(
                    steering
                )

            vehicle.step(
                steering,
                throttle,
            )

            if done: break

        print(
            f"Teacher episode "
            f"{episode + 1}/{episodes}"
        )

    return (
        np.asarray(
            X,
            dtype=np.float32,
        ),
        np.asarray(
            Y,
            dtype=np.float32,
        ),
    )


# ============================================================
# Readout Model
# ============================================================

def train_model(X, Y):
    model = make_pipeline(
        StandardScaler(),
        RidgeCV(
            alphas=[
                0.1,
                0.3,
                1.0,
                3.0,
                10.0,
                30.0,
                100.0,
            ],
            cv=3,
        ),
    )

    model.fit(
        X,
        Y,
    )

    pred = model.predict(
        X
    )

    rmse = np.sqrt(
        np.mean(
            (pred - Y) ** 2
        )
    )

    print(f"Samples: {len(X):,}")
    print(f"Train RMSE: {rmse:.4f}")
    print(f"Alpha: {model.named_steps['ridgecv'].alpha_}")

    return model


def predict_steering(model, feature):
    output = model.predict(
        feature[None]
    )[0]

    return float(
        np.tanh(output)
    )


# ============================================================
# DAgger
# ============================================================

def collect_dagger_data(
    model,
    brain,
    projector,
    encoder,
    episodes,
    steps,
    generator,
    beta,
):
    X, Y = [], []

    for episode in range(episodes):
        track = random_track(
            generator,
            aggressive=True,
        )

        vehicle = Vehicle(
            track,
            lateral_offset=generator.uniform(-2.5, 2.5),
            yaw_offset=generator.uniform(-0.45, 0.45),
        )

        brain.reset()

        steering = 0.0

        for t in range(steps):
            observation, done = observe(
                vehicle,
                track,
            )

            brain.step(
                encoder.encode(
                    observation
                )
            )

            feature = make_feature(
                brain,
                projector,
            )

            teacher = teacher_steering(
                observation
            )

            predicted = predict_steering(
                model,
                feature,
            )

            desired = (
                beta * teacher
                + (1 - beta) * predicted
            )

            if abs(observation[0]) > 5:
                desired = teacher

            diff = np.clip(
                desired - steering,
                -MAX_STEER_RATE,
                MAX_STEER_RATE,
            )

            steering += diff

            throttle = speed_controller(
                observation
            )

            if t >= WARMUP:
                X.append(
                    feature.copy()
                )

                Y.append(
                    teacher
                )

            vehicle.step(
                steering,
                throttle,
            )

            if done: break

        print(
            f"DAgger β={beta:.2f} "
            f"| {episode + 1}/{episodes}"
        )

    return (
        np.asarray(
            X,
            dtype=np.float32,
        ),
        np.asarray(
            Y,
            dtype=np.float32,
        ),
    )


def dagger_train(brain, projector, encoder):
    generator = np.random.default_rng(SEED)

    print("\n==============================")
    print("INITIAL TRAINING")
    print("==============================")

    X, Y = collect_teacher_data(
        brain,
        projector,
        encoder,
        TRAIN_EPISODES,
        TRAIN_STEPS,
        generator,
    )

    model = train_model(
        X,
        Y,
    )

    betas = [
        0.45,
        0.0,
    ]

    for i in range(DAGGER_ROUNDS):
        print("\n==============================")
        print(f"DAGGER {i + 1}/{DAGGER_ROUNDS}")
        print("==============================")

        X_new, Y_new = collect_dagger_data(
            model,
            brain,
            projector,
            encoder,
            DAGGER_EPISODES,
            DAGGER_STEPS,
            generator,
            betas[i],
        )

        X = np.concatenate([
            X,
            X_new,
        ])

        Y = np.concatenate([
            Y,
            Y_new,
        ])

        model = train_model(
            X,
            Y,
        )

    return model


# ============================================================
# Meaningful Neuron Scores
# ============================================================

def normalize_score(value):
    scale = np.percentile(
        value,
        99,
    )

    if scale <= 0:
        return np.zeros_like(
            value
        )

    return np.clip(
        value / scale,
        0,
        1,
    )


def readout_influence(model, projector):
    scaler = model.named_steps[
        "standardscaler"
    ]

    ridge = model.named_steps[
        "ridgecv"
    ]

    coef = (
        ridge.coef_
        / scaler.scale_
    )

    rate_coef = coef[
        :READOUT_DIM
    ]

    spike_coef = coef[
        READOUT_DIM:
        2 * READOUT_DIM
    ]

    rate = (
        rate_coef[
            projector.bucket
        ]
        * projector.sign
        / projector.norm[
            projector.bucket
        ]
    )

    spike = (
        spike_coef[
            projector.bucket
        ]
        * projector.sign
        / projector.norm[
            projector.bucket
        ]
    )

    return (
        np.abs(rate)
        + np.abs(spike)
    )


# ============================================================
# Profile Whole Brain
# ============================================================

def profile_brain(
    model,
    brain,
    projector,
    encoder,
    track,
    steps,
):
    vehicle = Vehicle(
        track,
        lateral_offset=1.4,
        yaw_offset=-0.18,
    )

    brain.reset()

    steering = 0.0

    rate_sum = np.zeros(
        brain.N,
        dtype=np.float64,
    )

    rate_sq_sum = np.zeros(
        brain.N,
        dtype=np.float64,
    )

    rate_steer_sum = np.zeros(
        brain.N,
        dtype=np.float64,
    )

    spike_count = np.zeros(
        brain.N,
        dtype=np.int64,
    )

    steering_sum = 0.0
    steering_sq_sum = 0.0

    count = 0

    print("\nProfiling whole brain...")

    for t in range(steps):
        observation, done = observe(
            vehicle,
            track,
        )

        brain.step(
            encoder.encode(
                observation
            )
        )

        feature = make_feature(
            brain,
            projector,
        )

        desired = predict_steering(
            model,
            feature,
        )

        diff = np.clip(
            desired - steering,
            -MAX_STEER_RATE,
            MAX_STEER_RATE,
        )

        steering += diff

        throttle = speed_controller(
            observation
        )

        vehicle.step(
            steering,
            throttle,
        )

        rate_sum += brain.rate
        rate_sq_sum += brain.rate**2
        rate_steer_sum += brain.rate * steering

        spike_count += brain.spike.astype(
            np.int64
        )

        steering_sum += steering
        steering_sq_sum += steering**2

        count += 1

        if (t + 1) % 100 == 0:
            print(
                f"\rProfile {t + 1}/{steps} "
                f"| spikes={int(brain.spike.sum()):,}",
                end="",
            )

        if done: break

    print()

    mean_rate = (
        rate_sum / count
    )

    variance = (
        rate_sq_sum / count
        - mean_rate**2
    )

    std_rate = np.sqrt(
        np.maximum(
            variance,
            0,
        )
    )

    mean_steering = (
        steering_sum / count
    )

    steering_var = (
        steering_sq_sum / count
        - mean_steering**2
    )

    steering_std = np.sqrt(
        max(
            steering_var,
            1e-8,
        )
    )

    covariance = (
        rate_steer_sum / count
        - mean_rate * mean_steering
    )

    correlation = np.abs(
        covariance
        / (
            std_rate
            * steering_std
            + 1e-8
        )
    )

    return (
        mean_rate,
        std_rate,
        spike_count,
        correlation,
    )


# ============================================================
# Select Meaningful Neurons
# ============================================================

def select_meaningful_neurons(
    model,
    projector,
    encoder,
    mean_rate,
    std_rate,
    spike_count,
    correlation,
):
    influence = readout_influence(
        model,
        projector,
    )

    activity_score = normalize_score(
        mean_rate
    )

    variation_score = normalize_score(
        std_rate
    )

    spike_score = normalize_score(
        np.log1p(
            spike_count
        )
    )

    influence_score = normalize_score(
        influence
    )

    correlation_score = np.clip(
        correlation,
        0,
        1,
    )

    score = (
        0.10 * activity_score
        + 0.25 * variation_score
        + 0.10 * spike_score
        + 0.30 * influence_score
        + 0.25 * correlation_score
    )

    score[
        spike_count == 0
    ] *= 0.02

    selected = np.argsort(
        -score
    )[:DISPLAY_NEURONS]

    steering_nodes = np.argsort(
        -influence
    )[:TOP_STEERING_NEURONS]

    selected = np.unique(
        np.concatenate([
            selected,
            encoder.nodes,
            steering_nodes,
        ])
    )

    print("\n==============================")
    print("TASK-RELEVANT NEURONS")
    print("==============================")

    print(f"Whole brain: {len(score):,}")
    print(f"Displayed: {len(selected):,}")
    print(f"Sensor nodes: {len(encoder.nodes):,}")
    print(f"Steering nodes: {len(steering_nodes):,}")

    return (
        selected,
        steering_nodes,
    )


# ============================================================
# Autonomous Simulation
# ============================================================

def simulate(
    model,
    brain,
    projector,
    encoder,
    track,
    display_nodes,
    steps,
):
    vehicle = Vehicle(
        track,
        lateral_offset=1.4,
        yaw_offset=-0.18,
    )

    brain.reset()

    steering = 0.0

    history = {
        "x": [],
        "y": [],
        "yaw": [],
        "speed": [],
        "cte": [],
        "heading": [],
        "curvature": [],
        "steering": [],
        "activity": [],
        "spike": [],
        "total_spikes": [],
        "track_idx": [],
    }

    for t in range(steps):
        observation, done = observe(
            vehicle,
            track,
        )

        brain.step(
            encoder.encode(
                observation
            )
        )

        feature = make_feature(
            brain,
            projector,
        )

        desired = predict_steering(
            model,
            feature,
        )

        diff = np.clip(
            desired - steering,
            -MAX_STEER_RATE,
            MAX_STEER_RATE,
        )

        steering += diff

        throttle = speed_controller(
            observation
        )

        vehicle.step(
            steering,
            throttle,
        )

        history["x"].append(
            vehicle.x
        )

        history["y"].append(
            vehicle.y
        )

        history["yaw"].append(
            vehicle.yaw
        )

        history["speed"].append(
            vehicle.v
        )

        history["cte"].append(
            observation[0]
        )

        history["heading"].append(
            observation[1]
        )

        history["curvature"].append(
            observation[2]
        )

        history["steering"].append(
            steering
        )

        history["activity"].append(
            brain.rate[
                display_nodes
            ].copy()
        )

        history["spike"].append(
            brain.spike[
                display_nodes
            ].copy()
        )

        history["total_spikes"].append(
            int(
                brain.spike.sum()
            )
        )

        history["track_idx"].append(
            vehicle.track_idx
        )

        if (t + 1) % 100 == 0:
            print(
                f"\rAutonomous {t + 1}/{steps} "
                f"| CTE={observation[0]:+.2f} m "
                f"| steer={steering:+.2f} "
                f"| spikes={int(brain.spike.sum()):,}",
                end="",
            )

        if done: break

    print()

    return {
        key: np.asarray(value)
        for key, value in history.items()
    }


# ============================================================
# Display Graph
# ============================================================

def build_display_graph(
    W,
    display_nodes,
):
    sub = W[
        display_nodes
    ][
        :,
        display_nodes
    ].tocoo()

    if len(sub.data) > DISPLAY_EDGES:
        idx = np.argpartition(
            np.abs(sub.data),
            -DISPLAY_EDGES,
        )[-DISPLAY_EDGES:]

        row = sub.row[idx]
        col = sub.col[idx]
        data = sub.data[idx]

    else:
        row = sub.row
        col = sub.col
        data = sub.data

    G = nx.DiGraph()

    G.add_nodes_from(
        range(
            len(display_nodes)
        )
    )

    for r, c, value in zip(
        row,
        col,
        data,
    ):
        G.add_edge(
            int(c),
            int(r),
            weight=float(value),
        )

    print(
        f"Displayed edges: "
        f"{G.number_of_edges():,}"
    )

    print(
        "Computing display layout..."
    )

    pos = nx.spring_layout(
        G,
        seed=SEED,
        k=0.16,
        iterations=70,
    )

    return G, pos


# ============================================================
# Save Vertical MP4
# ============================================================

def save_video(
    history,
    track,
    brain,
    encoder,
    display_nodes,
    steering_nodes,
    G,
    pos,
):
    frame_ids = np.arange(
        0,
        len(history["x"]),
        STEPS_PER_FRAME,
    )

    normal_x = -np.sin(
        track.yaw
    )

    normal_y = np.cos(
        track.yaw
    )

    left_x = (
        track.x
        + LANE_WIDTH / 2
        * normal_x
    )

    left_y = (
        track.y
        + LANE_WIDTH / 2
        * normal_y
    )

    right_x = (
        track.x
        - LANE_WIDTH / 2
        * normal_x
    )

    right_y = (
        track.y
        - LANE_WIDTH / 2
        * normal_y
    )

    # ========================================================
    # Vertical Layout
    # ========================================================

    fig, (
        ax_road,
        ax_brain,
    ) = plt.subplots(
        2,
        1,
        figsize=(10, 14),
    )

    fig.subplots_adjust(
        hspace=0.15
    )

    # ========================================================
    # Top: Track
    # ========================================================

    ax_road.plot(
        left_x,
        left_y,
        linewidth=2,
    )

    ax_road.plot(
        right_x,
        right_y,
        linewidth=2,
    )

    ax_road.plot(
        track.x,
        track.y,
        "--",
        linewidth=1,
    )

    car, = ax_road.plot(
        [],
        [],
        "o",
        markersize=11,
    )

    direction, = ax_road.plot(
        [],
        [],
        linewidth=2,
    )

    trail, = ax_road.plot(
        [],
        [],
        linewidth=1.5,
    )

    target, = ax_road.plot(
        [],
        [],
        "x",
        markersize=9,
    )

    ax_road.set_title(
        "MaleCNS Autonomous Driving"
    )

    ax_road.set_xlabel("x [m]"); ax_road.set_ylabel("y [m]")
    ax_road.grid(True); ax_road.set_aspect("equal")

    road_text = ax_road.text(
        0.02,
        0.97,
        "",
        transform=ax_road.transAxes,
        va="top",
    )

    # ========================================================
    # Bottom: Brain
    # ========================================================

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax_brain,
        arrows=False,
        alpha=0.07,
        width=0.5,
    )

    coords = np.asarray([
        pos[i]
        for i in range(
            len(display_nodes)
        )
    ])

    node_scatter = ax_brain.scatter(
        coords[:, 0],
        coords[:, 1],
        s=22,
        c=np.zeros(
            len(display_nodes)
        ),
        cmap="plasma",
        vmin=0,
        vmax=0.25,
    )

    sensor_mask = np.isin(
        display_nodes,
        encoder.nodes,
    )

    ax_brain.scatter(
        coords[sensor_mask, 0],
        coords[sensor_mask, 1],
        s=65,
        marker="s",
        facecolors="none",
        edgecolors="black",
        linewidths=1.3,
        label="Sensor",
    )

    steering_mask = np.isin(
        display_nodes,
        steering_nodes,
    )

    ax_brain.scatter(
        coords[steering_mask, 0],
        coords[steering_mask, 1],
        s=90,
        marker="^",
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        label="Steering influence",
    )

    ax_brain.set_title(
        "Task-Relevant MaleCNS Activity"
    )

    ax_brain.axis("off"); ax_brain.legend(loc="upper right")

    brain_text = ax_brain.text(
        0.02,
        0.97,
        "",
        transform=ax_brain.transAxes,
        va="top",
    )

    # ========================================================
    # Update
    # ========================================================

    def update(frame):
        i = frame_ids[frame]

        x = history["x"][i]
        y = history["y"][i]
        yaw = history["yaw"][i]

        car.set_data(
            [x],
            [y],
        )

        direction.set_data(
            [
                x,
                x + 3 * np.cos(yaw),
            ],
            [
                y,
                y + 3 * np.sin(yaw),
            ],
        )

        start = max(
            0,
            i - 250,
        )

        trail.set_data(
            history["x"][start:i + 1],
            history["y"][start:i + 1],
        )

        track_idx = history[
            "track_idx"
        ][i]

        lookahead_idx = min(
            track_idx + 25,
            track.N - 1,
        )

        target.set_data(
            [track.x[lookahead_idx]],
            [track.y[lookahead_idx]],
        )

        view = 22

        ax_road.set_xlim(
            x - view,
            x + view,
        )

        ax_road.set_ylim(
            y - view,
            y + view,
        )

        mean_cte = np.mean(
            np.abs(
                history["cte"][
                    max(0, i - 100):
                    i + 1
                ]
            )
        )

        road_text.set_text(
            f"t = {i * DT:.1f} s\n"
            f"speed = {history['speed'][i]:.2f} m/s\n"
            f"CTE = {history['cte'][i]:+.2f} m\n"
            f"heading = {np.rad2deg(history['heading'][i]):+.1f}°\n"
            f"curvature = {history['curvature'][i]:+.3f}\n"
            f"steering = {history['steering'][i]:+.2f}\n"
            f"mean |CTE| = {mean_cte:.2f} m"
        )

        activity = history[
            "activity"
        ][i]

        spike = history[
            "spike"
        ][i]

        node_scatter.set_array(
            activity
        )

        node_scatter.set_sizes(
            22
            + 110 * spike
        )

        brain_text.set_text(
            f"simulated neurons = {brain.N:,}\n"
            f"task-relevant shown = {len(display_nodes):,}\n"
            f"whole-brain spikes = {history['total_spikes'][i]:,}\n"
            f"shown active = {np.sum(activity > 0.02):,}"
        )

        return (
            car,
            direction,
            trail,
            target,
            node_scatter,
            road_text,
            brain_text,
        )

    anim = FuncAnimation(
        fig,
        update,
        frames=len(frame_ids),
        interval=int(
            DT
            * STEPS_PER_FRAME
            * 1000
        ),
        blit=False,
    )

    plt.tight_layout()

    fps = max(
        1,
        round(
            1
            / (
                DT
                * STEPS_PER_FRAME
            )
        ),
    )

    writer = FFMpegWriter(
        fps=fps,
        bitrate=4500,
        metadata={
            "title": "MaleCNS Vertical Autonomous Driving"
        },
    )

    print(
        f"\nSaving MP4: "
        f"{VIDEO_PATH}"
    )

    anim.save(
        VIDEO_PATH,
        writer=writer,
        dpi=130,
    )

    plt.close(fig)

    print(
        f"[OK] Saved: "
        f"{VIDEO_PATH}"
    )


# ============================================================
# Main
# ============================================================

download_dataset()

body_ids = scan_all_body_ids()
W = build_full_connectome(body_ids)

print("\n==============================")
print("FULL MALECNS")
print("==============================")

print(f"Simulated neurons: {W.shape[0]:,}")
print(f"Connections: {W.nnz:,}")

brain = FlyBrain(W)
encoder = FlyEncoder(W)
projector = BrainProjector(W.shape[0])


# ============================================================
# Train
# ============================================================

model = dagger_train(
    brain,
    projector,
    encoder,
)


# ============================================================
# Aggressive Test Track
# ============================================================

track = aggressive_test_track()


# ============================================================
# Profile Whole Brain
# ============================================================

(
    mean_rate,
    std_rate,
    spike_count,
    correlation,
) = profile_brain(
    model,
    brain,
    projector,
    encoder,
    track,
    TEST_STEPS,
)


# ============================================================
# Meaningful Neurons
# ============================================================

display_nodes, steering_nodes = select_meaningful_neurons(
    model,
    projector,
    encoder,
    mean_rate,
    std_rate,
    spike_count,
    correlation,
)


# ============================================================
# Autonomous Run
# ============================================================

print("\n==============================")
print("AUTONOMOUS RUN")
print("==============================")

history = simulate(
    model,
    brain,
    projector,
    encoder,
    track,
    display_nodes,
    TEST_STEPS,
)


# ============================================================
# Result
# ============================================================

cte = np.abs(
    history["cte"]
)

departure = np.mean(
    cte > LANE_WIDTH / 2
)

print("\n==============================")
print("RESULT")
print("==============================")

print(f"Mean |CTE|: {cte.mean():.3f} m")
print(f"Max |CTE|: {cte.max():.3f} m")
print(f"Lane departure: {departure * 100:.2f}%")
print(f"Distance index: {history['track_idx'][-1]}/{track.N - 1}")


# ============================================================
# Visualization Graph
# ============================================================

G, pos = build_display_graph(
    W,
    display_nodes,
)


# ============================================================
# Save MP4
# ============================================================

save_video(
    history,
    track,
    brain,
    encoder,
    display_nodes,
    steering_nodes,
    G,
    pos,
)


# ============================================================
# Notebook Video
# ============================================================

display(
    Video(
        str(VIDEO_PATH),
        embed=False,
    )
)